# 29 algorithmes de Machine Learning — notebook corrigé et enrichi

**Version Google Colab / GitHub — travail pédagogique basé sur `CC_Partie_A.ipynb` et `Consignes_ML.md`.**

- Les 29 démonstrations d’origine sont conservées, avec des corrections ciblées et des explications originales.
- Les jeux **Diabetes, Breast Cancer, Wine, Digits, Iris et Linnerud** sont des données publiques issues d’observations ; **les bandits et Taxi-v3 sont simulés**.
- Les sections finales produisent les **résultats en exécutant le code**, sans chiffres inventés. Les sorties sont à vérifier dans Colab avant le dépôt.
- Les formules emploient des notations usuelles ; les schémas sont générés dans ce notebook avec Matplotlib.
- Les résultats de petits échantillons ne sont pas généralisables ; aucune prédiction médicale n’est destinée à un usage clinique.

**À faire avant la remise :** exécuter `Exécution > Tout exécuter`, vérifier les sorties, compléter les noms, la date de consultation et les conditions de licence si demandées, puis enregistrer le notebook exécuté.


## Installation et reproductibilité

Exécuter la cellule suivante dans Google Colab. Les installations externes nécessitent une connexion Internet. La graine aléatoire est fixée à 42 quand c’est possible. Les versions sont affichées dans la cellule suivante.

In [ ]:
%pip -q install "scikit-learn>=1.3" "xgboost>=2.0" "umap-learn>=0.5" "gymnasium>=1.0"

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, sklearn, sys, platform
import importlib.metadata as md
from sklearn.datasets import load_diabetes, load_breast_cancer, load_wine, load_digits, load_iris, load_linnerud
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, f1_score,
    silhouette_score, accuracy_score, precision_score, recall_score, roc_auc_score,
    adjusted_rand_score, classification_report)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
GRAINE=42
np.random.seed(GRAINE)
for lib in ('numpy','pandas','scikit-learn','xgboost','umap-learn','gymnasium','matplotlib'):
    try: print(f'{lib}: {md.version(lib)}')
    except md.PackageNotFoundError: print(f'{lib}: non installé')
print('Python:',sys.version.split()[0])
jeu_reg=load_diabetes(); X_reg,y_reg=jeu_reg.data,jeu_reg.target
jeu_cls=load_breast_cancer(); X_cls,y_cls=jeu_cls.data,jeu_cls.target
jeu_groupes=load_wine()
X_clu=PCA(n_components=2,random_state=GRAINE).fit_transform(StandardScaler().fit_transform(jeu_groupes.data))
jeu_images=load_digits(); X_images=StandardScaler().fit_transform(jeu_images.data)
Xr_ent,Xr_test,yr_ent,yr_test=train_test_split(X_reg,y_reg,test_size=.25,random_state=GRAINE)
Xc_ent,Xc_test,yc_ent,yc_test=train_test_split(X_cls,y_cls,test_size=.25,stratify=y_cls,random_state=GRAINE)
print('Diabetes',X_reg.shape,'Breast cancer',X_cls.shape,'Wine',X_clu.shape,'Digits',X_images.shape)

## Q0 — Afficher et visualiser les données utilisées

**Correction de l’hypothèse de Q0 :** les exemples de régression, classification, clustering et réduction de dimension du notebook joint ne génèrent pas de données synthétiques ; ils chargent des jeux réels de scikit-learn. **Seuls les bandits** simulent des récompenses Bernoulli, et Taxi-v3 simule des interactions. On affiche donc les données réellement utilisées, et on visualise séparément une simulation de bandit.

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(12,8))
for ax,jeu,titre in zip(axes.flat,[jeu_reg,jeu_cls,jeu_groupes,jeu_images],['Diabetes','Breast Cancer','Wine','Digits']):
    X=np.asarray(jeu.data)
    print(f'\n{titre} — dimensions {X.shape}; cinq premières lignes :')
    display(pd.DataFrame(X,columns=jeu.feature_names).head())
    ax.scatter(X[:,0],X[:,1],c=jeu.target,cmap='viridis',s=15,alpha=.75)
    ax.set(xlabel=str(jeu.feature_names[0]),ylabel=str(jeu.feature_names[1]),title=titre)
plt.suptitle('Visualisations des données observées (2 variables seulement)')
plt.tight_layout();plt.show()
rng=np.random.default_rng(GRAINE)
probabilites_bandit=np.array([.15,.4,.6,.75])
tirages=(rng.random((200,4))<probabilites_bandit).astype(int)
print('Bandit — premières récompenses simulées :')
display(pd.DataFrame(tirages,columns=[f'Bras {i}' for i in range(4)]).head())
plt.figure(figsize=(7,3));plt.bar(range(4),tirages.mean(axis=0));plt.xticks(range(4));plt.ylim(0,1)
plt.ylabel('Gain moyen simulé');plt.xlabel('Bras');plt.title('Exemple de données synthétiques : bandit à 4 bras');plt.show()

## 1. Régression linéaire / Linear Regression

**Question ciblée du support :** Expliquez pourquoi une relation linéaire entre les variables médicales et la cible peut être insuffisante.

### Présentation

1. **Nom et famille :** Régression linéaire / Linear Regression — régression.
2. **Objectif :** Estime une relation additive entre les caractéristiques et une cible numérique.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** apprentissage des paramètres sur les données.
5. **Sortie :** prédictions sur de nouvelles observations.
6. **Critère essentiel :** $\hat y=\beta_0+\sum_j\beta_jx_j$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : coefficients β et constante β₀.
8. **Hyperparamètres fixés avant entraînement :** choix des variables.
9. **Deux avantages :** simple, interprétable.
10. **Deux limites :** sensibilité aux valeurs extrêmes, relations non linéaires.

**Définition des symboles :** $x$ est une observation, $y$ la valeur cible, $\hat y$ sa prédiction, $i$ indexe les observations, $j$ les variables, $\beta$ les coefficients ; $\alpha$ et $\eta$ sont des réglages lorsqu’ils apparaissent. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** Les flèches montrent la transformation des variables en prédiction numérique. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 1. Régression linéaire / Linear Regression
from matplotlib.patches import FancyBboxPatch
etapes=['Variables x', 'Ajustement des β', 'Somme pondérée', 'Valeur prédite']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('Régression linéaire / Linear Regression',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** prévision de ventes ;  estimation de dépenses médicales. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** relation fortement non linéaire non transformée.
- **Métriques adaptées :** MAE, RMSE, R². Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.linear_model import LinearRegression
modele=LinearRegression().fit(Xr_ent,yr_ent)
pred=modele.predict(Xr_test)
print('RMSE:',mean_squared_error(yr_test,pred)**.5,'R²:',r2_score(yr_test,pred))

**Commentaire des entrées, hyperparamètres et sorties :** Les entrées sont les 10 variables Diabetes ; LinearRegression n’a pas de pénalisation. RMSE et R² mesurent la performance sur les 25 % de test. Ils ne prouvent pas que les relations sont causales ou parfaitement linéaires.

**Réponse à la question ciblée :** Une droite suppose un effet additif constant : des seuils, interactions et courbures peuvent rester inexpliqués.

## 2. Ridge / Ridge Regression

**Question ciblée du support :** Que se passe-t-il lorsque la pénalisation L2 devient très forte ?

### Présentation

1. **Nom et famille :** Ridge / Ridge Regression — régression.
2. **Objectif :** Ajoute une pénalité quadratique pour limiter les coefficients excessifs.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** apprentissage des paramètres sur les données.
5. **Sortie :** prédictions sur de nouvelles observations.
6. **Critère essentiel :** $\min_\beta\sum_i(y_i-\hat y_i)^2+\alpha\sum_j\beta_j^2$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : coefficients β.
8. **Hyperparamètres fixés avant entraînement :** alpha (intensité L2).
9. **Deux avantages :** stabilité, résistance à la colinéarité.
10. **Deux limites :** ne sélectionne pas exactement les variables, dépend de l’échelle.

**Définition des symboles :** $x$ est une observation, $y$ la valeur cible, $\hat y$ sa prédiction, $i$ indexe les observations, $j$ les variables, $\beta$ les coefficients ; $\alpha$ et $\eta$ sont des réglages lorsqu’ils apparaissent. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** La pénalité L2 contracte les coefficients; alpha élevé peut sous-ajuster. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 2. Ridge / Ridge Regression
from matplotlib.patches import FancyBboxPatch
etapes=['Données standardisées', 'Erreur quadratique', 'Pénalité L2', 'Coefficients réduits']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('Ridge / Ridge Regression',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** prévision de coûts financiers ;  estimation énergétique. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** besoin de sélectionner exactement quelques variables.
- **Métriques adaptées :** MAE, RMSE, R². Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.linear_model import Ridge
modele=make_pipeline(StandardScaler(),Ridge(alpha=25.)).fit(Xr_ent,yr_ent)
print('MAE:',mean_absolute_error(yr_test,modele.predict(Xr_test)))

**Commentaire des entrées, hyperparamètres et sorties :** Ridge utilise une standardisation apprise sur le train, puis alpha=25. La MAE affichée mesure une erreur moyenne absolue sur le test. Une pénalisation trop forte réduit les coefficients et peut sous-ajuster.

**Réponse à la question ciblée :** Quand alpha augmente fortement, les coefficients se rapprochent de zéro et le modèle peut sous-ajuster.

## 3. Lasso / Lasso Regression

**Question ciblée du support :** Comment la pénalisation L1 sélectionne-t-elle des variables ?

### Présentation

1. **Nom et famille :** Lasso / Lasso Regression — régression.
2. **Objectif :** Réduit certains coefficients exactement à zéro grâce à une pénalité absolue.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** apprentissage des paramètres sur les données.
5. **Sortie :** prédictions sur de nouvelles observations.
6. **Critère essentiel :** $\min_\beta\sum_i(y_i-\hat y_i)^2+\alpha\sum_j|\beta_j|$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : coefficients β.
8. **Hyperparamètres fixés avant entraînement :** alpha (intensité L1).
9. **Deux avantages :** sélection automatique, modèle parcimonieux.
10. **Deux limites :** instable si variables corrélées, nécessite standardisation.

**Définition des symboles :** $x$ est une observation, $y$ la valeur cible, $\hat y$ sa prédiction, $i$ indexe les observations, $j$ les variables, $\beta$ les coefficients ; $\alpha$ et $\eta$ sont des réglages lorsqu’ils apparaissent. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** Les coefficients nuls indiquent les variables écartées par Lasso. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 3. Lasso / Lasso Regression
from matplotlib.patches import FancyBboxPatch
etapes=['Variables standardisées', 'Pénalité L1', 'Coefficients nuls', 'Prédiction']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('Lasso / Lasso Regression',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** sélection d’indicateurs boursiers ;  analyse de biomarqueurs. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** effets fortement non linéaires sans transformation.
- **Métriques adaptées :** MAE, RMSE, R². Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.linear_model import Lasso
modele=make_pipeline(StandardScaler(),Lasso(alpha=3.,max_iter=15000)).fit(Xr_ent,yr_ent)
print('R²:',r2_score(yr_test,modele.predict(Xr_test)))

**Commentaire des entrées, hyperparamètres et sorties :** Lasso utilise alpha=3 et max_iter=15000. Le R² est calculé sur le test. La pénalité L1 peut mettre certains coefficients à zéro ; le R² seul ne décrit pas la stabilité de cette sélection.

**Réponse à la question ciblée :** La pénalité L1 peut pousser des coefficients exactement à zéro : ces variables sont alors exclues du modèle.

## 4. Régression polynomiale / Polynomial Regression

**Question ciblée du support :** Quelle difficulté pose la croissance du nombre de caractéristiques polynomiales ?

### Présentation

1. **Nom et famille :** Régression polynomiale / Polynomial Regression — régression.
2. **Objectif :** Construit des puissances et interactions, puis ajuste une régression.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** apprentissage des paramètres sur les données.
5. **Sortie :** prédictions sur de nouvelles observations.
6. **Critère essentiel :** $\hat y=\beta_0+\beta_1x+\beta_2x^2$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : coefficients β.
8. **Hyperparamètres fixés avant entraînement :** degré polynomial, alpha de Ridge.
9. **Deux avantages :** capture la courbure, reste interprétable à petit degré.
10. **Deux limites :** explosion du nombre de termes, surapprentissage.

**Définition des symboles :** $x$ est une observation, $y$ la valeur cible, $\hat y$ sa prédiction, $i$ indexe les observations, $j$ les variables, $\beta$ les coefficients ; $\alpha$ et $\eta$ sont des réglages lorsqu’ils apparaissent. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** Le degré accroît la flexibilité et le risque de surapprentissage. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 4. Régression polynomiale / Polynomial Regression
from matplotlib.patches import FancyBboxPatch
etapes=['Variables x', 'Termes x² et interactions', 'Régression régularisée', 'Prédiction']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('Régression polynomiale / Polynomial Regression',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** modélisation des coûts de production ;  croissance biologique. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** extrapolation très éloignée des données.
- **Métriques adaptées :** MAE, RMSE, R². Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
modele=make_pipeline(StandardScaler(),PolynomialFeatures(degree=2,include_bias=False),Ridge(alpha=60.)).fit(Xr_ent,yr_ent)
print('MAE:',mean_absolute_error(yr_test,modele.predict(Xr_test)))

**Commentaire des entrées, hyperparamètres et sorties :** La transformation polynomiale degré 2 est ajustée dans un pipeline et suivie de Ridge. La MAE est hors échantillon ; ajouter beaucoup de termes augmente la complexité et le risque de surapprentissage.

**Réponse à la question ciblée :** Le nombre de termes augmente rapidement avec les variables et le degré, ce qui accroît coût, colinéarité et risque de surapprentissage.

## 5. Arbre de régression / Decision Tree Regressor

**Question ciblée du support :** Pourquoi limiter la profondeur et le nombre minimal d’observations par feuille ?

### Présentation

1. **Nom et famille :** Arbre de régression / Decision Tree Regressor — régression.
2. **Objectif :** Découpe l’espace en régions puis prédit une moyenne par feuille.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** apprentissage des paramètres sur les données.
5. **Sortie :** prédictions sur de nouvelles observations.
6. **Critère essentiel :** $\min_{s}\sum_{r\in\{g,d\}}\sum_{i\in r}(y_i-\bar y_r)^2$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : seuils et valeurs des feuilles.
8. **Hyperparamètres fixés avant entraînement :** max_depth, min_samples_leaf.
9. **Deux avantages :** non linéaire, lisible si peu profond.
10. **Deux limites :** instable, prédictions en paliers.

**Définition des symboles :** $x$ est une observation, $y$ la valeur cible, $\hat y$ sa prédiction, $i$ indexe les observations, $j$ les variables, $\beta$ les coefficients ; $\alpha$ et $\eta$ sont des réglages lorsqu’ils apparaissent. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** Chaque chemin de tests mène à une feuille avec une valeur moyenne. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 5. Arbre de régression / Decision Tree Regressor
from matplotlib.patches import FancyBboxPatch
etapes=['Variables', 'Tests de seuil', 'Branches', 'Moyenne en feuille']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('Arbre de régression / Decision Tree Regressor',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** estimation du prix immobilier ;  demande logistique. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** extrapolation continue hors intervalle observé.
- **Métriques adaptées :** MAE, RMSE, R². Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
modele=DecisionTreeRegressor(max_depth=3,min_samples_leaf=8,random_state=GRAINE).fit(Xr_ent,yr_ent)
print('R²:',r2_score(yr_test,modele.predict(Xr_test)))

**Commentaire des entrées, hyperparamètres et sorties :** L’arbre est limité à max_depth=3 et min_samples_leaf=8 ; le R² sur test mesure sa généralisation. Une profondeur limitée réduit la variance, mais peut empêcher d’apprendre certains motifs.

**Réponse à la question ciblée :** Une profondeur et une taille minimale de feuille contrôlées évitent des feuilles trop spécialisées et réduisent la variance.

## 6. Forêt aléatoire régression / Random Forest Regressor

**Question ciblée du support :** Décrivez le bootstrap et expliquez le rôle de la moyenne des arbres.

### Présentation

1. **Nom et famille :** Forêt aléatoire régression / Random Forest Regressor — régression.
2. **Objectif :** Moyenne des prédictions d’arbres entraînés sur des échantillons bootstrap.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** apprentissage des paramètres sur les données.
5. **Sortie :** prédictions sur de nouvelles observations.
6. **Critère essentiel :** $\hat y=\frac1B\sum_{b=1}^{B}T_b(x)$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : arbres et leurs seuils.
8. **Hyperparamètres fixés avant entraînement :** n_estimators, min_samples_leaf.
9. **Deux avantages :** robuste, capture interactions.
10. **Deux limites :** moins interprétable, plus coûteuse.

**Définition des symboles :** $x$ est une observation, $y$ la valeur cible, $\hat y$ sa prédiction, $i$ indexe les observations, $j$ les variables, $\beta$ les coefficients ; $\alpha$ et $\eta$ sont des réglages lorsqu’ils apparaissent. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** La moyenne réduit la variance par rapport à un arbre isolé. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 6. Forêt aléatoire régression / Random Forest Regressor
from matplotlib.patches import FancyBboxPatch
etapes=['Échantillons bootstrap', 'Arbres indépendants', 'Prédictions multiples', 'Moyenne']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('Forêt aléatoire régression / Random Forest Regressor',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** prévision de risque de crédit ;  estimation de consommation. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** extrapolation au-delà des cibles d’entraînement.
- **Métriques adaptées :** MAE, RMSE, R². Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
modele=RandomForestRegressor(n_estimators=120,min_samples_leaf=4,random_state=GRAINE,n_jobs=-1).fit(Xr_ent,yr_ent)
print('MAE:',mean_absolute_error(yr_test,modele.predict(Xr_test)))

**Commentaire des entrées, hyperparamètres et sorties :** La forêt utilise 120 arbres, des feuilles d’au moins 4 observations et une graine fixe. La MAE du test ne renseigne pas sur la causalité ou sur le comportement hors plage des cibles observées.

**Réponse à la question ciblée :** Le bootstrap crée des échantillons d’apprentissage différents ; la moyenne des arbres atténue leur variance.

## 7. Gradient Boosting régression / Gradient Boosting Regressor

**Question ciblée du support :** Pourquoi les arbres sont-ils entraînés séquentiellement dans le boosting ?

### Présentation

1. **Nom et famille :** Gradient Boosting régression / Gradient Boosting Regressor — régression.
2. **Objectif :** Ajoute séquentiellement des arbres qui corrigent les erreurs du modèle courant.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** apprentissage des paramètres sur les données.
5. **Sortie :** prédictions sur de nouvelles observations.
6. **Critère essentiel :** $F_m(x)=F_{m-1}(x)+\eta h_m(x)$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : arbres successifs.
8. **Hyperparamètres fixés avant entraînement :** learning_rate η, max_iter.
9. **Deux avantages :** bonne précision tabulaire, relations complexes.
10. **Deux limites :** réglage délicat, entraînement séquentiel.

**Définition des symboles :** $x$ est une observation, $y$ la valeur cible, $\hat y$ sa prédiction, $i$ indexe les observations, $j$ les variables, $\beta$ les coefficients ; $\alpha$ et $\eta$ sont des réglages lorsqu’ils apparaissent. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** Chaque arbre améliore la prédiction produite par les précédents. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 7. Gradient Boosting régression / Gradient Boosting Regressor
from matplotlib.patches import FancyBboxPatch
etapes=['Modèle initial', 'Résidus', 'Arbre correctif', 'Somme progressive']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('Gradient Boosting régression / Gradient Boosting Regressor',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** prévision de défaut de paiement en montant ;  demande commerciale. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** jeu minuscule avec fort bruit.
- **Métriques adaptées :** MAE, RMSE, R². Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
modele=HistGradientBoostingRegressor(max_iter=110,max_leaf_nodes=12,random_state=GRAINE).fit(Xr_ent,yr_ent)
print('RMSE:',mean_squared_error(yr_test,modele.predict(Xr_test))**.5)

**Commentaire des entrées, hyperparamètres et sorties :** HistGradientBoostingRegressor entraîne 110 itérations avec au plus 12 feuilles. La RMSE sur test pénalise davantage les grandes erreurs ; la comparaison doit être faite sur le même découpage.

**Réponse à la question ciblée :** Chaque nouvel arbre corrige les erreurs résiduelles du modèle précédent, d’où l’apprentissage séquentiel.

## 8. Régression logistique / Logistic Regression

**Question ciblée du support :** Comment interpréter une probabilité prédite et ajuster le seuil de classification ?

### Présentation

1. **Nom et famille :** Régression logistique / Logistic Regression — classification.
2. **Objectif :** Estime une probabilité de classe par une fonction sigmoïde.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** apprentissage des paramètres sur les données.
5. **Sortie :** prédictions sur de nouvelles observations.
6. **Critère essentiel :** $P(y=1|x)=\frac1{1+e^{-(\beta_0+\beta^Tx)}}$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : coefficients β.
8. **Hyperparamètres fixés avant entraînement :** C, seuil de décision.
9. **Deux avantages :** probabilités, interprétable.
10. **Deux limites :** frontière linéaire, dépend de l’échelle.

**Définition des symboles :** $x$ est une observation, $y$ une classe, $i$ une observation, $j$ une variable, $k$ une classe ; $P$ désigne une probabilité, $\beta$ des coefficients, $C$ et $\eta$ des réglages. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** Le seuil convertit la probabilité en décision de classe. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 8. Régression logistique / Logistic Regression
from matplotlib.patches import FancyBboxPatch
etapes=['Caractéristiques', 'Score linéaire', 'Sigmoïde', 'Seuil et classe']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('Régression logistique / Logistic Regression',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** détection de défaut de crédit ;  triage de messages. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** frontière très non linéaire sans transformations.
- **Métriques adaptées :** F1, rappel, précision, ROC-AUC. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.linear_model import LogisticRegression
modele=make_pipeline(StandardScaler(),LogisticRegression(C=.2,max_iter=1500,random_state=GRAINE)).fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))

**Commentaire des entrées, hyperparamètres et sorties :** La régression logistique utilise une standardisation, C=0.2 et max_iter=1500. Le F1 au seuil standard est calculé sur le test ; changer le seuil modifierait précision et rappel.

**Réponse à la question ciblée :** Une probabilité de 0,7 exprime un score estimé de classe positive ; changer le seuil de 0,5 à 0,7 diminue généralement les alertes et modifie précision/rappel.

## 9. KNN / k-Nearest Neighbors

**Question ciblée du support :** Quel est l’effet d’une distance normalisée et du nombre de voisins ?

### Présentation

1. **Nom et famille :** KNN / k-Nearest Neighbors — classification.
2. **Objectif :** Attribue une classe en consultant les k observations les plus proches.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** apprentissage des paramètres sur les données.
5. **Sortie :** prédictions sur de nouvelles observations.
6. **Critère essentiel :** $\hat y=\mathrm{mode}\{y_j:j\in N_k(x)\}$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : données mémorisées (pas de coefficients).
8. **Hyperparamètres fixés avant entraînement :** k, distance, weights.
9. **Deux avantages :** simple, non paramétrique.
10. **Deux limites :** coûteux en prédiction, sensible à l’échelle.

**Définition des symboles :** $x$ est une observation, $y$ une classe, $i$ une observation, $j$ une variable, $k$ une classe ; $P$ désigne une probabilité, $\beta$ des coefficients, $C$ et $\eta$ des réglages. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** Les voisins proches votent pour la classe du nouveau point. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 9. KNN / k-Nearest Neighbors
from matplotlib.patches import FancyBboxPatch
etapes=['Point à classer', 'Distances normalisées', 'k voisins', 'Vote pondéré']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('KNN / k-Nearest Neighbors',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** segmentation client ;  reconnaissance d’écriture. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** très haute dimension sans réduction.
- **Métriques adaptées :** F1, précision, rappel. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
modele=make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=9,weights='distance')).fit(Xc_ent,yc_ent)
print('Exactitude:',accuracy_score(yc_test,modele.predict(Xc_test)))

**Commentaire des entrées, hyperparamètres et sorties :** KNN utilise k=9 et des votes pondérés par distance après standardisation. L’exactitude affichée ne suffit pas si les classes sont déséquilibrées ; la distance dépend des variables retenues.

**Réponse à la question ciblée :** La standardisation évite qu’une variable à grande unité domine la distance ; petit k est plus flexible, grand k plus lisse.

## 10. Naive Bayes / Gaussian Naive Bayes

**Question ciblée du support :** Quelle hypothèse d’indépendance conditionnelle est faite par Naive Bayes ?

### Présentation

1. **Nom et famille :** Naive Bayes / Gaussian Naive Bayes — classification.
2. **Objectif :** Applique Bayes avec indépendance conditionnelle des caractéristiques.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** apprentissage des paramètres sur les données.
5. **Sortie :** prédictions sur de nouvelles observations.
6. **Critère essentiel :** $P(y|x)\propto P(y)\prod_jP(x_j|y)$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : moyennes, variances et a priori.
8. **Hyperparamètres fixés avant entraînement :** var_smoothing.
9. **Deux avantages :** rapide, peu de données.
10. **Deux limites :** hypothèse forte d’indépendance, probabilités parfois mal calibrées.

**Définition des symboles :** $x$ est une observation, $y$ une classe, $i$ une observation, $j$ une variable, $k$ une classe ; $P$ désigne une probabilité, $\beta$ des coefficients, $C$ et $\eta$ des réglages. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** Chaque variable contribue à la vraisemblance de chaque classe. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 10. Naive Bayes / Gaussian Naive Bayes
from matplotlib.patches import FancyBboxPatch
etapes=['Données par classe', 'Probabilités par variable', 'Produit conditionnel', 'Classe probable']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('Naive Bayes / Gaussian Naive Bayes',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** filtrage d’emails commerciaux ;  diagnostic exploratoire. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** variables très dépendantes pour interpréter les probabilités.
- **Métriques adaptées :** F1, précision, rappel. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.naive_bayes import GaussianNB
modele=GaussianNB(var_smoothing=1e-8).fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))

**Commentaire des entrées, hyperparamètres et sorties :** GaussianNB apprend des statistiques par classe et utilise var_smoothing=1e-8. Le F1 affiché ne vérifie pas l’indépendance conditionnelle supposée.

**Réponse à la question ciblée :** Naive Bayes suppose que les variables sont indépendantes conditionnellement à la classe, hypothèse rarement exacte.

## 11. Arbre de décision / Decision Tree Classifier

**Question ciblée du support :** Comment calculer l’impureté de Gini et choisir une séparation ?

### Présentation

1. **Nom et famille :** Arbre de décision / Decision Tree Classifier — classification.
2. **Objectif :** Sépare récursivement les données pour rendre les classes plus homogènes.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** apprentissage des paramètres sur les données.
5. **Sortie :** prédictions sur de nouvelles observations.
6. **Critère essentiel :** $G=1-\sum_kp_k^2$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : seuils, structure, feuilles.
8. **Hyperparamètres fixés avant entraînement :** max_depth, min_samples_leaf.
9. **Deux avantages :** explicable, traite non-linéarités.
10. **Deux limites :** variance élevée, surapprentissage.

**Définition des symboles :** $x$ est une observation, $y$ une classe, $i$ une observation, $j$ une variable, $k$ une classe ; $P$ désigne une probabilité, $\beta$ des coefficients, $C$ et $\eta$ des réglages. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** Un bon test réduit l’impureté des groupes enfants. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 11. Arbre de décision / Decision Tree Classifier
from matplotlib.patches import FancyBboxPatch
etapes=['Classes mélangées', 'Impureté de Gini', 'Tests binaires', 'Feuilles classées']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('Arbre de décision / Decision Tree Classifier',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** règles d’acceptation de crédit ;  qualité industrielle. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** petites variations de données sans régularisation.
- **Métriques adaptées :** F1, précision, rappel. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
modele=DecisionTreeClassifier(max_depth=3,min_samples_leaf=5,random_state=GRAINE).fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))

**Commentaire des entrées, hyperparamètres et sorties :** L’arbre de classification utilise max_depth=3 et min_samples_leaf=5. Le F1 mesure la qualité sur le test, mais l’arbre peut être sensible à un autre échantillonnage.

**Réponse à la question ciblée :** Gini vaut 1 moins la somme des probabilités de classe au carré ; on choisit la coupure qui réduit le plus l’impureté pondérée.

## 12. Forêt aléatoire classification / Random Forest Classifier

**Question ciblée du support :** Pourquoi une forêt est-elle généralement moins instable qu’un arbre seul ?

### Présentation

1. **Nom et famille :** Forêt aléatoire classification / Random Forest Classifier — classification.
2. **Objectif :** Agrège les votes d’arbres construits avec bootstrap et sous-ensembles de variables.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** apprentissage des paramètres sur les données.
5. **Sortie :** prédictions sur de nouvelles observations.
6. **Critère essentiel :** $\hat y=\mathrm{mode}\{T_b(x)\}_{b=1}^B$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : arbres et seuils.
8. **Hyperparamètres fixés avant entraînement :** n_estimators, max_depth.
9. **Deux avantages :** robuste, performante sur tabulaire.
10. **Deux limites :** moins lisible, calcul plus long.

**Définition des symboles :** $x$ est une observation, $y$ une classe, $i$ une observation, $j$ une variable, $k$ une classe ; $P$ désigne une probabilité, $\beta$ des coefficients, $C$ et $\eta$ des réglages. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** L’agrégation rend la décision moins sensible à un arbre particulier. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 12. Forêt aléatoire classification / Random Forest Classifier
from matplotlib.patches import FancyBboxPatch
etapes=['Bootstrap', 'Arbres diversifiés', 'Votes des arbres', 'Classe majoritaire']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('Forêt aléatoire classification / Random Forest Classifier',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** détection de fraude ;  reconnaissance de défauts. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** fortes contraintes d’explicabilité par règle unique.
- **Métriques adaptées :** F1, précision, rappel. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
modele=RandomForestClassifier(n_estimators=110,max_depth=6,random_state=GRAINE,n_jobs=-1).fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))

**Commentaire des entrées, hyperparamètres et sorties :** La forêt de classification utilise 110 arbres, max_depth=6 et une graine. Le F1 de test résume un compromis précision/rappel, sans garantir des probabilités bien calibrées.

**Réponse à la question ciblée :** La moyenne ou le vote de nombreux arbres réduit l’effet des variations d’un arbre individuel.

## 13. SVM / Support Vector Machine

**Question ciblée du support :** À quoi servent les vecteurs de support et le paramètre C ?

### Présentation

1. **Nom et famille :** SVM / Support Vector Machine — classification.
2. **Objectif :** Cherche une frontière à marge maximale entre les classes.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** apprentissage des paramètres sur les données.
5. **Sortie :** prédictions sur de nouvelles observations.
6. **Critère essentiel :** $\min_{w,b}\frac12\|w\|^2+C\sum_i\xi_i$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : vecteurs de support, coefficients.
8. **Hyperparamètres fixés avant entraînement :** C, noyau, gamma.
9. **Deux avantages :** efficace en dimension élevée, marge robuste.
10. **Deux limites :** échelle nécessaire, coûteux sur gros volumes.

**Définition des symboles :** $x$ est une observation, $y$ une classe, $i$ une observation, $j$ une variable, $k$ une classe ; $P$ désigne une probabilité, $\beta$ des coefficients, $C$ et $\eta$ des réglages. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** Les vecteurs de support déterminent la frontière séparatrice. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 13. SVM / Support Vector Machine
from matplotlib.patches import FancyBboxPatch
etapes=['Points étiquetés', 'Noyau / distances', 'Marge maximale', 'Frontière']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('SVM / Support Vector Machine',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** classification du risque ;  analyse d’images. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** millions d’observations sans approximation.
- **Métriques adaptées :** F1, précision, rappel. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.svm import SVC
modele=make_pipeline(StandardScaler(),SVC(kernel='rbf',C=2.,gamma='scale')).fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))

**Commentaire des entrées, hyperparamètres et sorties :** SVC utilise le noyau RBF, C=2 et gamma=scale après standardisation. Le F1 est hors échantillon ; le score ne montre pas directement la frontière ni la calibration des probabilités.

**Réponse à la question ciblée :** Les vecteurs de support déterminent la marge ; un C élevé pénalise fortement les erreurs, un C faible tolère davantage de violations.

## 14. Gradient Boosting classification / Gradient Boosting Classifier

**Question ciblée du support :** Comment corriger progressivement les erreurs de classification ?

### Présentation

1. **Nom et famille :** Gradient Boosting classification / Gradient Boosting Classifier — classification.
2. **Objectif :** Construit successivement des arbres qui réduisent une perte de classification.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** apprentissage des paramètres sur les données.
5. **Sortie :** prédictions sur de nouvelles observations.
6. **Critère essentiel :** $F_m=F_{m-1}+\eta h_m$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : arbres correctifs.
8. **Hyperparamètres fixés avant entraînement :** max_iter, max_leaf_nodes.
9. **Deux avantages :** précis, capture interactions.
10. **Deux limites :** sensibilité aux hyperparamètres, coût.

**Définition des symboles :** $x$ est une observation, $y$ une classe, $i$ une observation, $j$ une variable, $k$ une classe ; $P$ désigne une probabilité, $\beta$ des coefficients, $C$ et $\eta$ des réglages. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** Les corrections successives améliorent la fonction de score. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 14. Gradient Boosting classification / Gradient Boosting Classifier
from matplotlib.patches import FancyBboxPatch
etapes=['Prédiction initiale', 'Erreurs / gradient', 'Arbre correctif', 'Score final']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('Gradient Boosting classification / Gradient Boosting Classifier',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** score de crédit ;  prévision de churn. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** données très rares et bruitées.
- **Métriques adaptées :** F1, précision, rappel. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
modele=HistGradientBoostingClassifier(max_iter=100,max_leaf_nodes=10,random_state=GRAINE).fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))

**Commentaire des entrées, hyperparamètres et sorties :** Le boosting de classification utilise 100 itérations et 10 feuilles au plus. Le F1 sur test n’établit pas que tous les profils sont traités équitablement.

**Réponse à la question ciblée :** Chaque itération construit un nouvel arbre à partir du gradient de la perte, pour corriger les prédictions précédentes.

## 15. AdaBoost / Adaptive Boosting

**Question ciblée du support :** Comment AdaBoost repondère-t-il les exemples difficiles ?

### Présentation

1. **Nom et famille :** AdaBoost / Adaptive Boosting — classification.
2. **Objectif :** Accorde plus de poids aux observations difficiles au fil des classifieurs.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** apprentissage des paramètres sur les données.
5. **Sortie :** prédictions sur de nouvelles observations.
6. **Critère essentiel :** $F(x)=\sum_t\alpha_th_t(x)$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : classifieurs faibles et poids α.
8. **Hyperparamètres fixés avant entraînement :** n_estimators, learning_rate.
9. **Deux avantages :** simple, combine modèles faibles.
10. **Deux limites :** sensible au bruit, sensible aux outliers.

**Définition des symboles :** $x$ est une observation, $y$ une classe, $i$ une observation, $j$ une variable, $k$ une classe ; $P$ désigne une probabilité, $\beta$ des coefficients, $C$ et $\eta$ des réglages. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** Les erreurs augmentent l’importance des exemples difficiles. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 15. AdaBoost / Adaptive Boosting
from matplotlib.patches import FancyBboxPatch
etapes=['Poids initiaux', 'Classifieur faible', 'Repondération erreurs', 'Vote pondéré']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('AdaBoost / Adaptive Boosting',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** détection d’attrition ;  contrôle qualité. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** beaucoup d’étiquettes erronées.
- **Métriques adaptées :** F1, précision, rappel. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
modele=AdaBoostClassifier(n_estimators=90,learning_rate=.7,random_state=GRAINE).fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))

**Commentaire des entrées, hyperparamètres et sorties :** AdaBoost combine 90 classifieurs avec learning_rate=0.7. Le F1 est calculé sur le test ; le modèle peut être sensible aux exemples mal étiquetés.

**Réponse à la question ciblée :** AdaBoost accroît le poids des exemples mal classés, pour orienter les classifieurs suivants vers ces cas difficiles.

## 16. XGBoost / Extreme Gradient Boosting

**Question ciblée du support :** Quel est le rôle de la régularisation et du learning_rate dans XGBoost ?

### Présentation

1. **Nom et famille :** XGBoost / Extreme Gradient Boosting — classification.
2. **Objectif :** Optimise une perte régularisée en ajoutant des arbres de gradient.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** apprentissage des paramètres sur les données.
5. **Sortie :** prédictions sur de nouvelles observations.
6. **Critère essentiel :** $\mathcal L=\sum_i\ell(y_i,\hat y_i)+\sum_t\Omega(f_t)$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : arbres, scores de feuilles.
8. **Hyperparamètres fixés avant entraînement :** max_depth, learning_rate, reg_lambda.
9. **Deux avantages :** précision, régularisation.
10. **Deux limites :** complexité de réglage, moins explicable.

**Définition des symboles :** $x$ est une observation, $y$ une classe, $i$ une observation, $j$ une variable, $k$ une classe ; $P$ désigne une probabilité, $\beta$ des coefficients, $C$ et $\eta$ des réglages. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** La régularisation limite la complexité des arbres successifs. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 16. XGBoost / Extreme Gradient Boosting
from matplotlib.patches import FancyBboxPatch
etapes=['Gradient de perte', 'Nouvel arbre', 'Régularisation', 'Score cumulé']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('XGBoost / Extreme Gradient Boosting',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** défaut bancaire ;  prévision d’achat. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** trop peu de données pour une forte complexité.
- **Métriques adaptées :** F1, précision, rappel. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from xgboost import XGBClassifier
modele=XGBClassifier(n_estimators=100,max_depth=2,learning_rate=.09,random_state=GRAINE,eval_metric='logloss').fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))

**Commentaire des entrées, hyperparamètres et sorties :** XGBoost utilise 100 arbres, profondeur 2, learning_rate=0.09. Le F1 affiché est obtenu sur le test ; la performance dépend du découpage et des réglages.

**Réponse à la question ciblée :** La régularisation limite la complexité des arbres ; un learning_rate faible réduit l’impact de chaque nouvel arbre et demande souvent plus d’itérations.

## 17. K-means / K-means Clustering

**Question ciblée du support :** Comment affecter chaque vin à son centroïde puis recalculer celui-ci ?

### Présentation

1. **Nom et famille :** K-means / K-means Clustering — clustering.
2. **Objectif :** Affecte les points au centroïde le plus proche puis recalcule les centroïdes.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** estimation d’une structure ou de valeurs à partir des observations.
5. **Sortie :** groupes, projection, alertes ou décisions selon la méthode.
6. **Critère essentiel :** $\min_{\mu,z}\sum_i\|x_i-\mu_{z_i}\|^2$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : centroïdes μ et affectations z.
8. **Hyperparamètres fixés avant entraînement :** k, n_init.
9. **Deux avantages :** rapide, facile à expliquer.
10. **Deux limites :** suppose des groupes compacts, sensible aux valeurs extrêmes.

**Définition des symboles :** $x_i$ est le point $i$, $k$ le groupe, $\mu$ un centre, $z_i$ une affectation, $N_\varepsilon$ le voisinage de rayon $\varepsilon$, $\pi$ un poids et $\Sigma$ une covariance. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** On répète affectation et mise à jour jusqu’à stabilisation. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 17. K-means / K-means Clustering
from matplotlib.patches import FancyBboxPatch
etapes=['Points', 'k centroïdes', 'Affectation proche', 'Mise à jour μ']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('K-means / K-means Clustering',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** segmentation marketing ;  regroupement de vins. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** groupes non convexes et densités différentes.
- **Métriques adaptées :** silhouette, inertie, stabilité. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.cluster import KMeans
modele=KMeans(n_clusters=3,n_init=15,random_state=GRAINE).fit(X_clu)
print('Silhouette:',silhouette_score(X_clu,modele.labels_))
plt.scatter(X_clu[:,0],X_clu[:,1],c=modele.labels_,s=22);plt.title('K-means sur Wine (ACP)');plt.show()

**Commentaire des entrées, hyperparamètres et sorties :** K-means utilise k=3, 15 initialisations et une projection ACP de Wine. La silhouette évalue la séparation des groupes, pas leur concordance automatique avec les classes de vin.

**Réponse à la question ciblée :** Chaque point est affecté au centroïde le plus proche, puis chaque centroïde devient la moyenne des points de son groupe.

## 18. Classification hiérarchique / Agglomerative Clustering

**Question ciblée du support :** Que montre un dendrogramme et comment définit-on le lien de Ward ?

### Présentation

1. **Nom et famille :** Classification hiérarchique / Agglomerative Clustering — clustering.
2. **Objectif :** Fusionne progressivement les groupes les plus proches.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** estimation d’une structure ou de valeurs à partir des observations.
5. **Sortie :** groupes, projection, alertes ou décisions selon la méthode.
6. **Critère essentiel :** $\Delta(A,B)=\frac{|A||B|}{|A|+|B|}\|\mu_A-\mu_B\|^2$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : hiérarchie de fusions.
8. **Hyperparamètres fixés avant entraînement :** n_clusters, linkage.
9. **Deux avantages :** hiérarchie interprétable, pas de centres initiaux.
10. **Deux limites :** coût quadratique, choix du seuil.

**Définition des symboles :** $x_i$ est le point $i$, $k$ le groupe, $\mu$ un centre, $z_i$ une affectation, $N_\varepsilon$ le voisinage de rayon $\varepsilon$, $\pi$ un poids et $\Sigma$ une covariance. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** La hauteur des fusions représente leur dissimilarité. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 18. Classification hiérarchique / Agglomerative Clustering
from matplotlib.patches import FancyBboxPatch
etapes=['Points isolés', 'Distances entre groupes', 'Fusions successives', 'Dendrogramme']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('Classification hiérarchique / Agglomerative Clustering',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** segmentation de fournisseurs ;  taxonomie biologique. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** très grand jeu de données sans approximation.
- **Métriques adaptées :** silhouette, dendrogramme. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
etiquettes=AgglomerativeClustering(n_clusters=3,linkage='ward').fit_predict(X_clu)
print('Silhouette:',silhouette_score(X_clu,etiquettes))

**Commentaire des entrées, hyperparamètres et sorties :** Ward fusionne les groupes pour produire 3 clusters. La silhouette est interne ; sans dendrogramme, elle ne suffit pas à expliquer la hiérarchie.

**Réponse à la question ciblée :** Le dendrogramme représente l’ordre et la hauteur des fusions ; Ward fusionne les groupes entraînant la plus faible hausse de variance intra-groupe.

## 19. DBSCAN / Density-Based Spatial Clustering

**Question ciblée du support :** À quoi correspondent les points marqués comme bruit par DBSCAN ?

### Présentation

1. **Nom et famille :** DBSCAN / Density-Based Spatial Clustering — clustering.
2. **Objectif :** Forme des groupes de points densément connectés et marque le bruit.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** estimation d’une structure ou de valeurs à partir des observations.
5. **Sortie :** groupes, projection, alertes ou décisions selon la méthode.
6. **Critère essentiel :** $|N_\varepsilon(x)|\geq\mathrm{minPts}$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : groupes et points bruit.
8. **Hyperparamètres fixés avant entraînement :** eps, min_samples.
9. **Deux avantages :** formes irrégulières, détecte bruit.
10. **Deux limites :** paramètres délicats, densités variables.

**Définition des symboles :** $x_i$ est le point $i$, $k$ le groupe, $\mu$ un centre, $z_i$ une affectation, $N_\varepsilon$ le voisinage de rayon $\varepsilon$, $\pi$ un poids et $\Sigma$ une covariance. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** Les points non reliés à une région dense restent du bruit. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 19. DBSCAN / Density-Based Spatial Clustering
from matplotlib.patches import FancyBboxPatch
etapes=['Voisinage ε', 'Points cœur', 'Extension des groupes', 'Bruit = -1']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('DBSCAN / Density-Based Spatial Clustering',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** zones de livraison ;  zones d’activité géographique. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** densités très différentes sans adaptation.
- **Métriques adaptées :** nombre de groupes, silhouette hors bruit. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.cluster import DBSCAN
etiquettes=DBSCAN(eps=.7,min_samples=6).fit_predict(X_clu)
print('Groupes:',len(set(etiquettes)-{-1}),'Bruit:',int((etiquettes==-1).sum()))

**Commentaire des entrées, hyperparamètres et sorties :** DBSCAN utilise eps=0.7 et min_samples=6. Le nombre de groupes et de points de bruit est affiché ; « bruit » signifie atypique pour ces paramètres, pas une donnée fausse.

**Réponse à la question ciblée :** Un point de bruit (étiquette -1) n’appartient à aucun groupe dense pour les paramètres eps et min_samples choisis.

## 20. Mélanges gaussiens / Gaussian Mixture Models

**Question ciblée du support :** En quoi les appartenances probabilistes diffèrent-elles d’une affectation dure ?

### Présentation

1. **Nom et famille :** Mélanges gaussiens / Gaussian Mixture Models — clustering.
2. **Objectif :** Ajuste plusieurs distributions gaussiennes et calcule des appartenances probabilistes.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** estimation d’une structure ou de valeurs à partir des observations.
5. **Sortie :** groupes, projection, alertes ou décisions selon la méthode.
6. **Critère essentiel :** $p(x)=\sum_{k=1}^K\pi_k\mathcal N(x|\mu_k,\Sigma_k)$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : π, μ, matrices Σ.
8. **Hyperparamètres fixés avant entraînement :** n_components, covariance_type.
9. **Deux avantages :** probabilités souples, groupes elliptiques.
10. **Deux limites :** hypothèse gaussienne, minima locaux.

**Définition des symboles :** $x_i$ est le point $i$, $k$ le groupe, $\mu$ un centre, $z_i$ une affectation, $N_\varepsilon$ le voisinage de rayon $\varepsilon$, $\pi$ un poids et $\Sigma$ une covariance. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** Un point peut appartenir partiellement à plusieurs groupes. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 20. Mélanges gaussiens / Gaussian Mixture Models
from matplotlib.patches import FancyBboxPatch
etapes=['Nuage de points', 'Composantes gaussiennes', 'Probabilités d’appartenance', 'Classe probable']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('Mélanges gaussiens / Gaussian Mixture Models',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** segmentation de clientèle ;  profils de qualité. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** groupes très non gaussiens.
- **Métriques adaptées :** BIC, silhouette, vraisemblance. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.mixture import GaussianMixture
modele=GaussianMixture(n_components=3,covariance_type='full',random_state=GRAINE).fit(X_clu)
etiquettes=modele.predict(X_clu)
print('Poids des groupes:',modele.weights_.round(3),'Silhouette:',silhouette_score(X_clu,etiquettes))

**Commentaire des entrées, hyperparamètres et sorties :** GaussianMixture utilise 3 composantes et une covariance complète. Les appartenances sont probabilistes ; une silhouette ne suffit pas à valider la forme gaussienne.

**Réponse à la question ciblée :** Le GMM fournit une probabilité par composante pour chaque point ; K-means attribue une seule classe sans incertitude.

## 21. ACP / Principal Component Analysis

**Question ciblée du support :** Que mesure la variance expliquée par les deux axes de l’ACP ?

### Présentation

1. **Nom et famille :** ACP / Principal Component Analysis — réduction de dimension.
2. **Objectif :** Projette les données sur des directions maximisant la variance.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** estimation d’une structure ou de valeurs à partir des observations.
5. **Sortie :** groupes, projection, alertes ou décisions selon la méthode.
6. **Critère essentiel :** $\max_{\|v\|=1}\mathrm{Var}(Xv)$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : axes principaux v.
8. **Hyperparamètres fixés avant entraînement :** n_components.
9. **Deux avantages :** rapide, variance expliquée.
10. **Deux limites :** linéaire, axes sensibles à l’échelle.

**Définition des symboles :** $X$ désigne les données, $v$ une direction, $Y$ les coordonnées projetées, $P$ et $Q$ des similarités, $G$ un graphe ; $KL$ est la divergence de Kullback–Leibler, $CE$ une entropie croisée. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** Les deux premiers axes résument une partie de la variance. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 21. ACP / Principal Component Analysis
from matplotlib.patches import FancyBboxPatch
etapes=['Données centrées', 'Matrice de covariance', 'Axes principaux', 'Projection 2D']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('ACP / Principal Component Analysis',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** compression d’indicateurs financiers ;  visualisation de capteurs. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** structure essentiellement non linéaire.
- **Métriques adaptées :** variance expliquée, trustworthiness. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.decomposition import PCA
modele=PCA(n_components=2,random_state=GRAINE).fit(X_images)
projection=modele.transform(X_images)
print('Variance expliquée:',modele.explained_variance_ratio_.round(3))
plt.scatter(projection[:,0],projection[:,1],c=jeu_images.target,cmap='tab10',s=5);plt.title('ACP sur Digits');plt.show()

**Commentaire des entrées, hyperparamètres et sorties :** L’ACP projette Digits en 2 axes. Les ratios de variance expliquée indiquent la part de variance conservée ; la figure ne démontre pas une séparation parfaite des chiffres.

**Réponse à la question ciblée :** La somme des ratios de variance des deux axes indique la part de variance totale capturée par le plan ACP.

## 22. t-SNE / t-distributed Stochastic Neighbor Embedding

**Question ciblée du support :** Pourquoi la distance entre groupes éloignés peut-elle être trompeuse en t-SNE ?

### Présentation

1. **Nom et famille :** t-SNE / t-distributed Stochastic Neighbor Embedding — réduction de dimension.
2. **Objectif :** Projette les observations en préservant surtout leurs voisinages locaux.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** estimation d’une structure ou de valeurs à partir des observations.
5. **Sortie :** groupes, projection, alertes ou décisions selon la méthode.
6. **Critère essentiel :** $\min_Y KL(P\|Q)$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : positions 2D Y.
8. **Hyperparamètres fixés avant entraînement :** perplexity, learning_rate.
9. **Deux avantages :** visualisation locale, groupes visibles.
10. **Deux limites :** distances globales trompeuses, pas de transformation simple.

**Définition des symboles :** $X$ désigne les données, $v$ une direction, $Y$ les coordonnées projetées, $P$ et $Q$ des similarités, $G$ un graphe ; $KL$ est la divergence de Kullback–Leibler, $CE$ une entropie croisée. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** La carte privilégie les voisinages, pas les distances globales. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 22. t-SNE / t-distributed Stochastic Neighbor Embedding
from matplotlib.patches import FancyBboxPatch
etapes=['Distances initiales', 'Similarités P', 'Optimisation KL', 'Carte 2D']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('t-SNE / t-distributed Stochastic Neighbor Embedding',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** exploration de segments clients ;  visualisation d’images. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** interpréter les distances entre groupes comme métriques.
- **Métriques adaptées :** trustworthiness, inspection. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.manifold import TSNE
projection=TSNE(n_components=2,perplexity=35,random_state=GRAINE,init='pca').fit_transform(X_images[:600])
plt.scatter(projection[:,0],projection[:,1],c=jeu_images.target[:600],cmap='tab10',s=8);plt.title('t-SNE sur Digits');plt.show()

**Commentaire des entrées, hyperparamètres et sorties :** t-SNE utilise 600 chiffres, perplexity=35, init=pca et une graine fixe. La figure représente des voisinages locaux ; les distances entre groupes éloignés ne doivent pas être surinterprétées.

**Réponse à la question ciblée :** t-SNE optimise surtout les voisinages proches ; l’écart entre deux amas éloignés sur la carte n’est pas une distance fiable dans l’espace initial.

## 23. UMAP / Uniform Manifold Approximation and Projection

**Question ciblée du support :** Quel est l’effet de n_neighbors et min_dist sur les voisinages ?

### Présentation

1. **Nom et famille :** UMAP / Uniform Manifold Approximation and Projection — réduction de dimension.
2. **Objectif :** Construit un graphe de voisins puis cherche une représentation compacte.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** estimation d’une structure ou de valeurs à partir des observations.
5. **Sortie :** groupes, projection, alertes ou décisions selon la méthode.
6. **Critère essentiel :** $\min_Y\mathrm{CE}(G_X,G_Y)$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : coordonnées projetées Y.
8. **Hyperparamètres fixés avant entraînement :** n_neighbors, min_dist.
9. **Deux avantages :** rapide, préserve localement.
10. **Deux limites :** résultats sensibles aux paramètres, distances globales incertaines.

**Définition des symboles :** $X$ désigne les données, $v$ une direction, $Y$ les coordonnées projetées, $P$ et $Q$ des similarités, $G$ un graphe ; $KL$ est la divergence de Kullback–Leibler, $CE$ une entropie croisée. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** Le graphe conserve des relations locales entre observations. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 23. UMAP / Uniform Manifold Approximation and Projection
from matplotlib.patches import FancyBboxPatch
etapes=['Voisins kNN', 'Graphe pondéré', 'Optimisation 2D', 'Carte UMAP']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('UMAP / Uniform Manifold Approximation and Projection',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** cartographie clients ;  exploration d’images. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** estimer un effet causal depuis une projection.
- **Métriques adaptées :** trustworthiness, inspection. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
import umap
projection=umap.UMAP(n_components=2,n_neighbors=20,min_dist=.15,random_state=GRAINE).fit_transform(X_images[:600])
plt.scatter(projection[:,0],projection[:,1],c=jeu_images.target[:600],cmap='tab10',s=8);plt.title('UMAP sur Digits');plt.show()

**Commentaire des entrées, hyperparamètres et sorties :** UMAP utilise 600 chiffres, n_neighbors=20 et min_dist=0.15. La carte est exploratoire ; l’espacement des groupes dépend des hyperparamètres.

**Réponse à la question ciblée :** n_neighbors plus petit privilégie le détail local ; min_dist plus petit permet des amas plus serrés dans la projection.

## 24. Forêt d’isolement / Isolation Forest

**Question ciblée du support :** Pourquoi une observation statistiquement atypique n’est-elle pas forcément une erreur ?

### Présentation

1. **Nom et famille :** Forêt d’isolement / Isolation Forest — détection d’anomalies.
2. **Objectif :** Isole les observations rares par des partitions aléatoires.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** estimation d’une structure ou de valeurs à partir des observations.
5. **Sortie :** groupes, projection, alertes ou décisions selon la méthode.
6. **Critère essentiel :** $s(x,n)=2^{-E[h(x)]/c(n)}$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : arbres aléatoires.
8. **Hyperparamètres fixés avant entraînement :** contamination, n_estimators.
9. **Deux avantages :** pas besoin de labels, efficace.
10. **Deux limites :** alertes pas forcément erreurs, dépend des paramètres.

**Définition des symboles :** $x$ est une observation, $h(x)$ la profondeur d’isolement, $E$ son espérance, $n$ la taille de l’échantillon, $c(n)$ une normalisation. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** Un chemin court suggère un point plus facile à isoler. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 24. Forêt d’isolement / Isolation Forest
from matplotlib.patches import FancyBboxPatch
etapes=['Échantillons', 'Coupes aléatoires', 'Longueur de chemin', 'Score anomalie']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('Forêt d’isolement / Isolation Forest',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** transactions atypiques ;  surveillance industrielle. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** déclarer une fraude sans vérification humaine.
- **Métriques adaptées :** nombre d’alertes, précision/rappel si labels fiables. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
from sklearn.ensemble import IsolationForest
# Recherche d'observations atypiques dans les caractéristiques de vins (sans ajouter de faux points)
modele=IsolationForest(contamination=.05,n_estimators=150,random_state=GRAINE).fit(X_clu)
indices=np.where(modele.predict(X_clu)==-1)[0]
print('Nombre de vins atypiques:',len(indices),'Indices (extrait):',indices[:12])

**Commentaire des entrées, hyperparamètres et sorties :** Isolation Forest marque environ 5 % des vins projetés comme atypiques selon contamination=0.05. Ce ne sont pas nécessairement des erreurs ni des fraudes : aucune étiquette d’anomalie fiable n’est fournie.

**Réponse à la question ciblée :** Un point rare peut être légitime : la rareté statistique ne prouve ni erreur, ni fraude, ni maladie.

## 25. Bandits manchots / Multi-Armed Bandits

**Question ciblée du support :** Comment calculer une estimation du gain moyen de chaque bras ?

### Présentation

1. **Nom et famille :** Bandits manchots / Multi-Armed Bandits — renforcement.
2. **Objectif :** Teste plusieurs actions pour estimer leurs récompenses moyennes.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** estimation d’une structure ou de valeurs à partir des observations.
5. **Sortie :** groupes, projection, alertes ou décisions selon la méthode.
6. **Critère essentiel :** $\hat\mu_a=\frac{\sum_t r_t1(a_t=a)}{N_a}$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : estimations μ et compteurs.
8. **Hyperparamètres fixés avant entraînement :** stratégie de sélection, horizon.
9. **Deux avantages :** simple, apprentissage en ligne.
10. **Deux limites :** récompenses retardées difficiles, exploration coûteuse.

**Définition des symboles :** $s$ est l’état, $a$ l’action, $r$ la récompense, $Q$ la valeur état-action, $\alpha$ le pas d’apprentissage, $\gamma$ le facteur d’actualisation, $\varepsilon$ la probabilité d’exploration, $d$ indique une fin d’épisode ; $t$ indexe les étapes. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** Les gains sont simulés ici, pas issus d’une base réelle. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 25. Bandits manchots / Multi-Armed Bandits
from matplotlib.patches import FancyBboxPatch
etapes=['Choisir un bras', 'Observer récompense', 'Mettre à jour μ', 'Comparer les bras']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('Bandits manchots / Multi-Armed Bandits',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** test de promotions ;  sélection d’annonces. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** décisions à états complexes sans contexte.
- **Métriques adaptées :** récompense cumulée, regret. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
# Bandit simulé à 4 choix : la politique choisit successivement chaque bras
generateur=np.random.default_rng(GRAINE)
probabilites=np.array([.15,.4,.6,.75]);sommes=np.zeros(4);comptes=np.zeros(4,dtype=int)
for t in range(800):
    bras=t%4
    gain=int(generateur.random()<probabilites[bras]);sommes[bras]+=gain;comptes[bras]+=1
print('Probabilités estimées:',(sommes/comptes).round(3))

**Commentaire des entrées, hyperparamètres et sorties :** Le bandit simule 800 tirages Bernoulli sur quatre bras avec probabilités prédéfinies et sélection cyclique. Les moyennes estimées sont calculées ; ces données sont synthétiques et ne valident pas une politique réelle.

**Réponse à la question ciblée :** Pour chaque bras, on divise la somme des gains observés par le nombre de fois où ce bras a été choisi.

## 26. Epsilon-greedy / Epsilon-Greedy

**Question ciblée du support :** Quelle différence entre exploration aléatoire et exploitation du meilleur bras estimé ?

### Présentation

1. **Nom et famille :** Epsilon-greedy / Epsilon-Greedy — renforcement.
2. **Objectif :** Explore au hasard avec probabilité epsilon, sinon choisit l’action estimée meilleure.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** estimation d’une structure ou de valeurs à partir des observations.
5. **Sortie :** groupes, projection, alertes ou décisions selon la méthode.
6. **Critère essentiel :** $a_t=\mathrm{random}\ (\varepsilon),\ \arg\max_a\hat\mu_a\ (1-\varepsilon)$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : estimations de récompense.
8. **Hyperparamètres fixés avant entraînement :** epsilon, horizon.
9. **Deux avantages :** simple, équilibre exploration/exploitation.
10. **Deux limites :** exploration parfois inutile, epsilon fixe sous-optimal.

**Définition des symboles :** $s$ est l’état, $a$ l’action, $r$ la récompense, $Q$ la valeur état-action, $\alpha$ le pas d’apprentissage, $\gamma$ le facteur d’actualisation, $\varepsilon$ la probabilité d’exploration, $d$ indique une fin d’épisode ; $t$ indexe les étapes. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** Le tirage contrôle le compromis entre exploration et exploitation. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 26. Epsilon-greedy / Epsilon-Greedy
from matplotlib.patches import FancyBboxPatch
etapes=['Tirage aléatoire', 'Explorer si u<ε', 'Sinon exploiter', 'Actualiser gains']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('Epsilon-greedy / Epsilon-Greedy',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** optimisation d’offres commerciales ;  choix de contenu. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** fort coût de toute action exploratoire.
- **Métriques adaptées :** récompense cumulée, regret. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
# Epsilon-greedy : exploration et exploitation des 4 bras simulés
rng=np.random.default_rng(GRAINE)
probabilites=np.array([.15,.4,.6,.75]);sommes=np.zeros(4);comptes=np.zeros(4,dtype=int);epsilon=.15
for t in range(1200):
    if t<4: bras=t  # chaque bras est observé au moins une fois
    elif rng.random()<epsilon: bras=int(rng.integers(4))
    else: bras=int(np.argmax(sommes/comptes))
    gain=int(rng.random()<probabilites[bras]);sommes[bras]+=gain;comptes[bras]+=1
print('Nombre de choix:',comptes,'Estimations:',(sommes/np.maximum(comptes,1)).round(3))

**Commentaire des entrées, hyperparamètres et sorties :** Epsilon-greedy simule 1200 choix avec epsilon=0.15. Après un essai initial par bras, il explore aléatoirement ou exploite la meilleure moyenne empirique ; les gains restent simulés.

**Réponse à la question ciblée :** L’exploration teste un bras au hasard ; l’exploitation choisit le bras dont le gain moyen estimé est le plus élevé.

## 27. Q-learning / Q-Learning

**Question ciblée du support :** Comment calculer une cible de Bellman dans Taxi et pourquoi le max intervient-il ?

### Présentation

1. **Nom et famille :** Q-learning / Q-Learning — renforcement.
2. **Objectif :** Apprend la valeur optimale état-action via une cible utilisant le maximum futur.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** estimation d’une structure ou de valeurs à partir des observations.
5. **Sortie :** groupes, projection, alertes ou décisions selon la méthode.
6. **Critère essentiel :** $Q\leftarrow Q+\alpha[r+\gamma\max_{a\prime}Q(s\prime,a\prime)-Q]$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : table Q.
8. **Hyperparamètres fixés avant entraînement :** alpha, gamma, epsilon.
9. **Deux avantages :** apprend hors politique, concept clair.
10. **Deux limites :** table immense si états nombreux, simulation requise.

**Définition des symboles :** $s$ est l’état, $a$ l’action, $r$ la récompense, $Q$ la valeur état-action, $\alpha$ le pas d’apprentissage, $\gamma$ le facteur d’actualisation, $\varepsilon$ la probabilité d’exploration, $d$ indique une fin d’épisode ; $t$ indexe les étapes. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** La cible prend la meilleure valeur future estimée, indépendamment de l’action réellement choisie. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 27. Q-learning / Q-Learning
from matplotlib.patches import FancyBboxPatch
etapes=['État s', 'Choix action a', 'Récompense et s suivant', 'Mise à jour max Q']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('Q-learning / Q-Learning',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** routage logistique simplifié ;  Taxi-v3. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** états continus très nombreux sans approximation.
- **Métriques adaptées :** récompense cumulée, taux de réussite. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
# Q-learning manuel sur Taxi-v3 : exploration epsilon-greedy
import gymnasium as gym
env=gym.make('Taxi-v3');Q=np.zeros((env.observation_space.n,env.action_space.n));generateur=np.random.default_rng(GRAINE)
for episode in range(3000):
    etat,_=env.reset(seed=GRAINE+episode);epsilon=max(.07,1-episode/2400)
    for pas in range(200):
        action=int(generateur.integers(env.action_space.n)) if generateur.random()<epsilon else int(np.argmax(Q[etat]))
        suivant,recompense,termine,tronque,_=env.step(action)
        cible=recompense+(0 if (termine or tronque) else .96*np.max(Q[suivant]))
        Q[etat,action]+=.2*(cible-Q[etat,action]);etat=suivant
        if termine or tronque:break
print('Q-learning Taxi — taille de Q:',Q.shape,'Valeurs du premier état:',Q[0].round(2));env.close()

**Commentaire des entrées, hyperparamètres et sorties :** Q-learning entraîne une table sur Taxi-v3, avec alpha=0.2, gamma=0.96 et epsilon décroissant. L’affichage de Q ne suffit pas à prouver la réussite : il faut tester la politique sur de nouveaux épisodes.

**Réponse à la question ciblée :** La cible est récompense plus gamma fois la meilleure valeur estimée à l’état suivant ; max correspond à une politique gloutonne future.

## 28. SARSA / State-Action-Reward-State-Action

**Question ciblée du support :** Pourquoi SARSA met-il à jour Q avec l’action suivante réellement choisie ?

### Présentation

1. **Nom et famille :** SARSA / State-Action-Reward-State-Action — renforcement.
2. **Objectif :** Apprend une valeur état-action avec la prochaine action effectivement sélectionnée.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** estimation d’une structure ou de valeurs à partir des observations.
5. **Sortie :** groupes, projection, alertes ou décisions selon la méthode.
6. **Critère essentiel :** $Q\leftarrow Q+\alpha[r+\gamma Q(s\prime,a\prime)-Q]$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : table Q.
8. **Hyperparamètres fixés avant entraînement :** alpha, gamma, epsilon.
9. **Deux avantages :** sur politique, tient compte de l’exploration.
10. **Deux limites :** peut apprendre lentement, table peu scalable.

**Définition des symboles :** $s$ est l’état, $a$ l’action, $r$ la récompense, $Q$ la valeur état-action, $\alpha$ le pas d’apprentissage, $\gamma$ le facteur d’actualisation, $\varepsilon$ la probabilité d’exploration, $d$ indique une fin d’épisode ; $t$ indexe les étapes. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** SARSA utilise l’action suivante effectivement choisie, contrairement au maximum de Q-learning. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 28. SARSA / State-Action-Reward-State-Action
from matplotlib.patches import FancyBboxPatch
etapes=['État et action', 'Récompense', 'Action suivante réelle', 'Mise à jour SARSA']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('SARSA / State-Action-Reward-State-Action',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** planification d’itinéraires ;  Taxi-v3. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** états d’image bruts sans approximation.
- **Métriques adaptées :** récompense cumulée, taux de réussite. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
# SARSA manuel sur Taxi-v3 (on-policy)
import gymnasium as gym
env=gym.make('Taxi-v3');Q=np.zeros((env.observation_space.n,env.action_space.n));generateur=np.random.default_rng(GRAINE)
def choisir_action(etat,epsilon):
    return int(generateur.integers(env.action_space.n)) if generateur.random()<epsilon else int(np.argmax(Q[etat]))
for episode in range(3000):
    etat,_=env.reset(seed=GRAINE+episode);epsilon=max(.07,1-episode/2400);action=choisir_action(etat,epsilon)
    for pas in range(200):
        suivant,recompense,termine,tronque,_=env.step(action)
        action_suiv=choisir_action(suivant,epsilon)
        cible=recompense+(0 if (termine or tronque) else .96*Q[suivant,action_suiv])
        Q[etat,action]+=.2*(cible-Q[etat,action]);etat,action=suivant,action_suiv
        if termine or tronque:break
print('SARSA Taxi — taille de Q:',Q.shape,'Valeurs du premier état:',Q[0].round(2));env.close()

**Commentaire des entrées, hyperparamètres et sorties :** SARSA entraîne une table sur Taxi-v3 avec action suivante effectivement choisie. Les valeurs de Q ne sont pas directement des taux de réussite ; il faut une évaluation sur épisodes distincts.

**Réponse à la question ciblée :** SARSA évalue la politique réellement suivie, exploration comprise, et utilise donc l’action suivante effectivement tirée.

## 29. DQN / Deep Q-Network

**Question ciblée du support :** Pourquoi un DQN complet a-t-il besoin d’un réseau cible et d’une mémoire de rejouement ?

### Présentation

1. **Nom et famille :** DQN / Deep Q-Network — renforcement.
2. **Objectif :** Approxime Q par un réseau et stabilise l’apprentissage avec mémoire et réseau cible.
3. **Entrée :** matrice de caractéristiques X (et étiquettes y si supervisé).
4. **Calcul :** estimation d’une structure ou de valeurs à partir des observations.
5. **Sortie :** groupes, projection, alertes ou décisions selon la méthode.
6. **Critère essentiel :** $y=r+\gamma(1-d)\max_{a\prime}Q_{cible}(s\prime,a\prime)$.
7. **Symboles :** voir la note ci-dessous ; paramètres appris : poids du réseau.
8. **Hyperparamètres fixés avant entraînement :** gamma, taille mémoire, learning_rate.
9. **Deux avantages :** gère états riches, approximation.
10. **Deux limites :** entraînement instable, calcul coûteux.

**Définition des symboles :** $s$ est l’état, $a$ l’action, $r$ la récompense, $Q$ la valeur état-action, $\alpha$ le pas d’apprentissage, $\gamma$ le facteur d’actualisation, $\varepsilon$ la probabilité d’exploration, $d$ indique une fin d’épisode ; $t$ indexe les étapes. Les paramètres sont estimés à partir des données ; les hyperparamètres sont fixés avant apprentissage.

### Schéma explicatif

**Schéma original généré ci-dessous.** Le code fourni calcule seulement des cibles; il n’entraîne aucun réseau. Les boîtes représentent des étapes successives et les flèches le flux d’information.

In [ ]:
# Schéma pédagogique original — 29. DQN / Deep Q-Network
from matplotlib.patches import FancyBboxPatch
etapes=['Transition (s,a,r,s’)', 'Mémoire replay', 'Réseau cible', 'Cible Bellman']
fig,ax=plt.subplots(figsize=(11,2.1))
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for j,libelle in enumerate(etapes):
    x=.1+j*2.55
    ax.add_patch(FancyBboxPatch((x,.55),2.25,.85,boxstyle='round,pad=0.1',fill=False,linewidth=1.5))
    ax.text(x+1.125,.97,libelle,ha='center',va='center',fontsize=9,wrap=True)
    if j<3: ax.annotate('',xy=(x+2.52,.97),xytext=(x+2.37,.97),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('DQN / Deep Q-Network',fontsize=12)
plt.tight_layout();plt.show()

### Contextes d'utilisation

- **Applications concrètes :** allocation de ressources ;  jeux Atari. La première application relève de la finance, du marketing, de la gestion ou de la logistique.
- **Données et prérequis :** tableau numérique ou données converties en variables exploitables ; traitement des valeurs manquantes et contrôle de l’échelle si les distances ou la régularisation sont utilisées. Pour le renforcement, un environnement fournissant états, actions et récompenses est indispensable.
- **Peu approprié lorsque :** présenter une simple cible comme un DQN entraîné.
- **Métriques adaptées :** récompense cumulée, taux de réussite. Une métrique seule ne démontre pas une causalité ni une validité hors échantillon.

### Analyse de l’expérience Python

**Exécuter le code d’origine conservé ci-dessous.** Les chiffres affichés sont calculés à l’exécution, non recopiés ni supposés.

In [ ]:
# DQN : calcul pédagogique d'une cible de Bellman (aucun réseau entraîné)
recompenses=np.array([2.,-1.,10.,0.]);termine=np.array([False,False,True,False])
q_cible_etat_suivant=np.array([5.,3.,7.,2.]);gamma=.9
cibles=recompenses+gamma*(~termine)*q_cible_etat_suivant
print('Cibles DQN conceptuelles:',cibles)
# Un DQN complet utiliserait un réseau en ligne, un réseau cible et un replay buffer.

**Commentaire des entrées, hyperparamètres et sorties :** Le calcul montre des cibles de Bellman sur des nombres pédagogiques inventés. Aucun réseau neuronal, replay buffer ni DQN complet n’est entraîné ; il est incorrect de présenter cela comme un résultat DQN.

**Réponse à la question ciblée :** Le replay buffer casse les corrélations entre transitions ; le réseau cible ralentit les variations de la cible et stabilise l’apprentissage.

## Annexe — implémentations manuelles (codes d’origine conservés)

Ces exercices illustrent les calculs ; ils ne remplacent pas les validations train/validation/test. Les comparaisons d’inertie K-means peuvent différer selon l’initialisation.

In [ ]:
# Descente de gradient manuelle pour une régression linéaire sur Diabetes, avec 1 variable
x=X_reg[:,2].copy();x=(x-x.mean())/x.std();y=y_reg.copy()
poids=0.;biais=y.mean();pas=.015
for iteration in range(3000):
    erreur=poids*x+biais-y
    poids-=pas*2*np.mean(erreur*x)
    biais-=pas*2*np.mean(erreur)
print('Manuel:',round(poids,3),round(biais,3))
from sklearn.linear_model import LinearRegression
modele_ref=LinearRegression().fit(x.reshape(-1,1),y)
print('Scikit-learn:',modele_ref.coef_.round(3),round(modele_ref.intercept_,3))

In [ ]:
# K-means manuel sur la projection Wine, avec centroïdes et inertie
valeurs=X_clu.copy();k=3;generateur=np.random.default_rng(GRAINE)
centroides=valeurs[generateur.choice(len(valeurs),size=k,replace=False)].copy()
for iteration in range(100):
    distances=((valeurs[:,None,:]-centroides[None,:,:])**2).sum(axis=2)
    groupes=distances.argmin(axis=1)
    nouveaux=np.array([valeurs[groupes==j].mean(axis=0) if np.any(groupes==j) else centroides[j] for j in range(k)])
    if np.allclose(nouveaux,centroides):
        centroides=nouveaux
        break
    centroides=nouveaux
groupes=((valeurs[:,None,:]-centroides[None,:,:])**2).sum(axis=2).argmin(axis=1)
inertie=np.sum((valeurs-centroides[groupes])**2)
from sklearn.cluster import KMeans
reference=KMeans(n_clusters=k,n_init=15,random_state=GRAINE).fit(valeurs)
print('Inertie manuelle:',round(inertie,2),'Inertie sklearn:',round(reference.inertia_,2))
# Attention : K-means est sensible à l'initialisation ; les deux inerties peuvent différer.

# Q5–Q8 — Expérimentations finales sur des données observées

**Protocole :** pour les méthodes supervisées, entraînement / validation / test ; le test n’est évalué qu’une fois avec les réglages choisis sur la validation. Le prétraitement appris est intégré aux pipelines. Pour les méthodes non supervisées, les étiquettes ne servent pas à construire les groupes.

**Provenance et traçabilité (à vérifier avant la remise) :**

| Famille | Données | Organisme et URL officielle | Taille et cible | Conditions |
|---|---|---|---|---|
| Régression | Auto MPG (priorité) | UCI, https://archive.ics.uci.edu/dataset/9/auto+mpg | 398 lignes brutes, cible `mpg`, 7 prédicteurs numériques retenus | Vérifier la fiche UCI et la licence avant remise ; données chargées en ligne |
| Régression (repli hors ligne) | Linnerud | scikit-learn, https://scikit-learn.org/stable/datasets/toy_dataset.html#linnerud-dataset | 20 lignes, 3 prédicteurs, cible `Waist` | Usage éducatif ; petit échantillon, conclusions très fragiles |
| Classification | Wine | scikit-learn / UCI, https://archive.ics.uci.edu/dataset/109/wine | 178 lignes, 13 variables, 3 classes | Consulter la licence de la fiche source |
| Clustering | Iris | scikit-learn / UCI, https://archive.ics.uci.edu/dataset/53/iris | 150 lignes, 4 variables, sans cible utilisée pour entraîner | Consulter la licence de la fiche source |
| Réduction | Wine | scikit-learn / UCI, https://archive.ics.uci.edu/dataset/109/wine | 178 lignes, 13 variables | Consulter la licence de la fiche source |
| Anomalies | Diabetes | scikit-learn, https://scikit-learn.org/stable/datasets/toy_dataset.html#diabetes-dataset | 442 lignes, 10 variables, aucune étiquette fiable d’anomalie | Usage éducatif, ne pas interpréter comme diagnostic |
| Renforcement | Taxi-v3 | Gymnasium, https://gymnasium.farama.org/environments/toy_text/taxi/ | environnement simulé, pas de CSV réel | Simulation d’interaction, explicitement distincte des données réelles |

**Date de consultation des sources :** à inscrire après votre propre vérification des pages et de leur licence. Les tailles effectives et colonnes sont affichées par le code ; les données manquantes d’Auto MPG sont retirées uniquement après inspection. La partie renforcement ne répond pas à une exigence de données réelles au sens observationnel : il s’agit d’une exception méthodologique prévue dans les consignes.


In [ ]:
# Q5–Q6 : charger un jeu de régression réel distinct du cas Diabetes de démonstration
URL_AUTO_MPG='https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'
colonnes_auto=['mpg','cylinders','displacement','horsepower','weight','acceleration','model_year','origin','car_name']
try:
    auto=pd.read_csv(URL_AUTO_MPG,sep=r'\s+',names=colonnes_auto[:8],usecols=range(8),na_values='?',engine='python')
    print('Auto MPG brut:',auto.shape,'\nManquants:',auto.isna().sum().to_dict(),'\nDoublons:',auto.duplicated().sum())
    auto=auto.dropna().drop_duplicates()
    X_final_reg=auto.drop(columns=['mpg']); y_final_reg=auto['mpg']
    source_reg='Auto MPG — UCI (lignes incomplètes retirées)'
except Exception as e:
    print('Téléchargement Auto MPG indisponible:',str(e)[:150])
    print('REPLI : Linnerud, 20 observations seulement — interprétation prudente.')
    lin=load_linnerud()
    X_final_reg=pd.DataFrame(lin.data,columns=lin.feature_names)
    y_final_reg=pd.Series(lin.target[:,2],name='Waist')
    source_reg='Linnerud — scikit-learn (repli hors ligne)'
print('Source retenue:',source_reg,'; X:',X_final_reg.shape,'; y:',y_final_reg.shape)
display(pd.DataFrame(X_final_reg).head())
print('Types:',X_final_reg.dtypes.astype(str).to_dict())
print('Doublons:',pd.DataFrame(X_final_reg).duplicated().sum())
print('Cible, statistiques:');display(pd.Series(y_final_reg).describe())


In [ ]:
# Découpage 60 / 20 / 20 et fonctions d'évaluation : aucun ajustement sur le test
from sklearn.base import clone
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier

def diviser_3(X,y,stratify=False):
    strat=y if stratify else None
    Xa,Xtest,ya,ytest=train_test_split(X,y,test_size=.20,random_state=GRAINE,stratify=strat)
    strat_a=ya if stratify else None
    Xtrain,Xval,ytrain,yval=train_test_split(Xa,ya,test_size=.25,random_state=GRAINE,stratify=strat_a)
    return Xtrain,Xval,Xtest,ytrain,yval,ytest

def metriques_reg(y,pred):
    return {'MAE':mean_absolute_error(y,pred),'RMSE':np.sqrt(mean_squared_error(y,pred)),'R2':r2_score(y,pred)}
Xtr,Xva,Xte,ytr,yva,yte=diviser_3(X_final_reg,y_final_reg)
modeles_reg={
 'Linéaire':make_pipeline(StandardScaler(),LinearRegression()),
 'Ridge':make_pipeline(StandardScaler(),Ridge(alpha=10)),
 'Lasso':make_pipeline(StandardScaler(),Lasso(alpha=.1,max_iter=15000)),
 'Polynomiale + Ridge':make_pipeline(StandardScaler(),PolynomialFeatures(degree=2,include_bias=False),Ridge(alpha=10)),
 'Arbre':DecisionTreeRegressor(max_depth=3,min_samples_leaf=3,random_state=GRAINE),
 'Forêt aléatoire':RandomForestRegressor(n_estimators=100,min_samples_leaf=2,random_state=GRAINE,n_jobs=-1),
 'Boosting':HistGradientBoostingRegressor(max_iter=80,max_leaf_nodes=10,min_samples_leaf=3,random_state=GRAINE)
}
val_reg=[]
for nom,modele in modeles_reg.items():
    try:
        modele.fit(Xtr,ytr);v=metriques_reg(yva,modele.predict(Xva));val_reg.append({'Modèle':nom,**v})
    except Exception as e: print('Échec',nom,str(e)[:100])
res_val_reg=pd.DataFrame(val_reg).sort_values('RMSE')
print('Validation régression —',source_reg);display(res_val_reg.round(3))
choix_reg=res_val_reg.iloc[0]['Modèle']
modele_final_reg=clone(modeles_reg[choix_reg]).fit(pd.concat([pd.DataFrame(Xtr),pd.DataFrame(Xva)]),pd.concat([pd.Series(ytr),pd.Series(yva)]))
res_test_reg=metriques_reg(yte,modele_final_reg.predict(Xte))
print('Modèle retenu sur validation :',choix_reg,'— test final:',{k:round(v,3) for k,v in res_test_reg.items()})
plt.figure(figsize=(9,4));plt.bar(res_val_reg['Modèle'],res_val_reg['RMSE']);plt.xticks(rotation=45,ha='right');plt.ylabel('RMSE validation');plt.title('Régression — comparaison hors test');plt.tight_layout();plt.show()

In [ ]:
# Classification réelle : Wine, 3 classes, F1 macro et rappel macro
vin=load_wine(as_frame=True)
Xc=vin.data;yc=vin.target
print('Wine :',Xc.shape,'classes :',yc.value_counts().to_dict(),'manquants :',Xc.isna().sum().sum(),'doublons :',Xc.duplicated().sum())
C_tr,C_va,C_te,cy_tr,cy_va,cy_te=diviser_3(Xc,yc,stratify=True)
modeles_cls={
 'Logistique':make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,random_state=GRAINE)),
 'KNN':make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=7)),
 'Naive Bayes':GaussianNB(),
 'Arbre':DecisionTreeClassifier(max_depth=4,random_state=GRAINE),
 'Forêt':RandomForestClassifier(n_estimators=100,random_state=GRAINE,n_jobs=-1),
 'SVM':make_pipeline(StandardScaler(),SVC(C=1,kernel='rbf',probability=True,random_state=GRAINE)),
 'Boosting':HistGradientBoostingClassifier(max_iter=80,min_samples_leaf=5,random_state=GRAINE),
 'AdaBoost':AdaBoostClassifier(n_estimators=80,random_state=GRAINE),
 'XGBoost':XGBClassifier(n_estimators=80,max_depth=3,learning_rate=.08,random_state=GRAINE,eval_metric='mlogloss')
}
res_cls=[]
for nom,modele in modeles_cls.items():
    try:
        modele.fit(C_tr,cy_tr);pred=modele.predict(C_va)
        res_cls.append({'Modèle':nom,'F1 macro':f1_score(cy_va,pred,average='macro'),
                        'Précision macro':precision_score(cy_va,pred,average='macro',zero_division=0),
                        'Rappel macro':recall_score(cy_va,pred,average='macro',zero_division=0)})
    except Exception as e:print('Échec',nom,str(e)[:100])
res_val_cls=pd.DataFrame(res_cls).sort_values('F1 macro',ascending=False)
display(res_val_cls.round(3))
choix_cls=res_val_cls.iloc[0]['Modèle']
modele_final_cls=clone(modeles_cls[choix_cls]).fit(pd.concat([C_tr,C_va]),pd.concat([cy_tr,cy_va]))
print('Meilleur sur validation:',choix_cls,'\nTEST final :\n',classification_report(cy_te,modele_final_cls.predict(C_te),zero_division=0))
plt.figure(figsize=(9,4));plt.bar(res_val_cls['Modèle'],res_val_cls['F1 macro']);plt.xticks(rotation=45,ha='right');plt.ylim(0,1.05)
plt.ylabel('F1 macro validation');plt.title('Classification Wine : comparaison');plt.tight_layout();plt.show()

In [ ]:
# Clustering réel : Iris (sans utiliser les vraies classes pour ajuster les groupes)
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
iris=load_iris(as_frame=True)
Xi=iris.data
print('Iris :',Xi.shape,'manquants :',Xi.isna().sum().sum(),'doublons :',Xi.duplicated().sum())
Xi_scaled=StandardScaler().fit_transform(Xi)  # non supervisé : aucun test prédictif ici
clusters={
 'K-means':KMeans(n_clusters=3,n_init=15,random_state=GRAINE).fit_predict(Xi_scaled),
 'Hiérarchique Ward':AgglomerativeClustering(n_clusters=3,linkage='ward').fit_predict(Xi_scaled),
 'DBSCAN':DBSCAN(eps=.8,min_samples=5).fit_predict(Xi_scaled),
 'GMM':GaussianMixture(n_components=3,random_state=GRAINE).fit_predict(Xi_scaled)
}
rows=[]
for nom,lab in clusters.items():
    mask=lab!=-1
    ngrp=len(set(lab[mask]))
    score=silhouette_score(Xi_scaled[mask],lab[mask]) if ngrp>=2 and ngrp<mask.sum() else np.nan
    rows.append({'Méthode':nom,'Groupes (hors bruit)':ngrp,'Bruit':int((~mask).sum()),'Silhouette hors bruit':score})
res_clu=pd.DataFrame(rows);display(res_clu.round(3))
# Stabilité K-means : comparaison des affectations obtenues avec plusieurs graines
base=KMeans(n_clusters=3,n_init=1,random_state=42).fit_predict(Xi_scaled)
stabilite=[adjusted_rand_score(base,KMeans(n_clusters=3,n_init=1,random_state=s).fit_predict(Xi_scaled)) for s in (7,21,99)]
print('Stabilité K-means (ARI entre graines) :',np.round(stabilite,3),'moyenne:',round(np.mean(stabilite),3))
coords=PCA(n_components=2,random_state=GRAINE).fit_transform(Xi_scaled)
fig,axes=plt.subplots(1,4,figsize=(15,3.5))
for ax,(nom,lab) in zip(axes,clusters.items()):
    ax.scatter(coords[:,0],coords[:,1],c=lab,cmap='tab10',s=18);ax.set_title(nom)
plt.suptitle('Iris — groupes calculés sans les vraies étiquettes');plt.tight_layout();plt.show()

In [ ]:
# Réduction de dimension réelle : Wine, PCA vs t-SNE vs UMAP
from sklearn.manifold import TSNE, trustworthiness
import umap
Xred=StandardScaler().fit_transform(load_wine().data)
yred=load_wine().target # couleurs seulement, jamais utilisé pour ajuster
pca=PCA(n_components=2,random_state=GRAINE).fit(Xred)
embeddings={
 'ACP':pca.transform(Xred),
 't-SNE':TSNE(n_components=2,perplexity=25,random_state=GRAINE,init='pca').fit_transform(Xred),
 'UMAP':umap.UMAP(n_components=2,n_neighbors=15,min_dist=.15,random_state=GRAINE).fit_transform(Xred)
}
red_results=pd.DataFrame([{'Méthode':nom,'Trustworthiness (k=10)':trustworthiness(Xred,xy,n_neighbors=10)} for nom,xy in embeddings.items()])
print('Variance expliquée ACP (2 axes):',round(pca.explained_variance_ratio_.sum(),3))
display(red_results.round(3))
fig,axes=plt.subplots(1,3,figsize=(12,3.5))
for ax,(nom,xy) in zip(axes,embeddings.items()):
    ax.scatter(xy[:,0],xy[:,1],c=yred,cmap='tab10',s=18);ax.set_title(nom)
plt.suptitle('Wine — projection des mêmes observations');plt.tight_layout();plt.show()
print('Attention : la variance expliquée ne s’applique pas à t-SNE et UMAP comme à l’ACP.')

In [ ]:
# Anomalies sur Diabetes : alertes exploratoires, PAS de vérités terrain d’anomalies
from sklearn.ensemble import IsolationForest
Xd=load_diabetes().data
modele_anom=make_pipeline(StandardScaler(),IsolationForest(n_estimators=150,contamination=.05,random_state=GRAINE))
alertes=modele_anom.fit_predict(Xd)
print('Alertes :',int(np.sum(alertes==-1)),'sur',len(alertes),'observations')
print('Sans labels fiables d’anomalies, précision et rappel ne sont PAS calculables.')
coords_d=PCA(n_components=2,random_state=GRAINE).fit_transform(StandardScaler().fit_transform(Xd))
plt.figure(figsize=(7,4));plt.scatter(coords_d[:,0],coords_d[:,1],c=np.where(alertes==-1,1,0),cmap='coolwarm',s=18)
plt.title('Diabetes — points atypiques signalés par Isolation Forest');plt.xlabel('ACP 1');plt.ylabel('ACP 2');plt.show()

In [ ]:
# Renforcement : comparaison Q-learning et SARSA sur l’environnement SIMULÉ Taxi-v3
import gymnasium as gym

def entrainer_taxi(methode='Q-learning',seed=42,episodes=1000):
    env=gym.make('Taxi-v3');rng=np.random.default_rng(seed)
    Q=np.zeros((env.observation_space.n,env.action_space.n))
    def politique(s,eps):
        return int(rng.integers(env.action_space.n)) if rng.random()<eps else int(np.argmax(Q[s]))
    for ep in range(episodes):
        s,_=env.reset(seed=seed+ep);eps=max(.05,1-ep/(episodes*.85));a=politique(s,eps)
        for _ in range(200):
            sp,r,term,trunc,_=env.step(a)
            ap=politique(sp,eps)
            futur=0 if term or trunc else (.96*np.max(Q[sp]) if methode=='Q-learning' else .96*Q[sp,ap])
            Q[s,a]+=.2*(r+futur-Q[s,a]);s,a=sp,ap
            if term or trunc:break
    # Evaluation séparée sans exploration
    gains=[];succes=[]
    for j in range(80):
        s,_=env.reset(seed=seed+100000+j);total=0;term=False
        for _ in range(200):
            s,r,term,trunc,_=env.step(int(np.argmax(Q[s])));total+=r
            if term or trunc:break
        gains.append(total);succes.append(int(term))
    env.close()
    return np.mean(gains),np.mean(succes)
res_rl=[]
for methode in ('Q-learning','SARSA'):
    for seed in (7,21,42):
        gain,succes=entrainer_taxi(methode,seed)
        res_rl.append({'Méthode':methode,'Graine':seed,'Récompense moyenne':gain,'Taux de réussite':succes})
res_rl=pd.DataFrame(res_rl)
display(res_rl.round(3))
display(res_rl.groupby('Méthode')[['Récompense moyenne','Taux de réussite']].agg(['mean','std']).round(3))
print('DQN : uniquement cible conceptuelle dans la section 29 ; aucun entraînement ni score DQN.')
res_rl.groupby('Méthode')['Récompense moyenne'].mean().plot(kind='bar',title='Taxi-v3 — récompense moyenne par méthode',ylabel='Récompense moyenne')
plt.xticks(rotation=0);plt.tight_layout();plt.show()

## Q8 — Interprétation et limites des comparaisons

Les tableaux et graphiques ci-dessus sont produits **après exécution**. Pour la régression et la classification, le modèle retenu est celui qui obtient le meilleur résultat sur **validation**, puis il est réentraîné sur entraînement + validation et testé **une seule fois**. Le tableau des autres modèles est une comparaison sur validation, **pas une comparaison de performances sur le test final**.

- **Régression :** comparer MAE, RMSE et R², mais un jeu de 20 lignes (repli Linnerud) ne permet pas de conclusions robustes.
- **Classification :** F1 macro accorde un poids égal aux trois classes de Wine. Les écarts peuvent varier selon la partition.
- **Clustering :** la silhouette est interne ; la stabilité de K-means est illustrée par l’ARI entre plusieurs initialisations. DBSCAN exclut le bruit de la silhouette affichée.
- **Réduction :** la variance expliquée concerne l’ACP ; trustworthiness mesure la préservation locale pour les trois projections.
- **Anomalies :** les points signalés ne sont pas des fraudes ou des erreurs confirmées ; sans vérité terrain, aucune précision/rappel n’est affirmée.
- **Renforcement :** Taxi-v3 est une simulation publique ; les moyennes et écarts-types sont calculés sur plusieurs graines. Un DQN complet reste facultatif et **n’a pas été réalisé**.

**Coût et interprétabilité :** les modèles linéaires sont souvent plus faciles à interpréter ; les ensembles d’arbres peuvent améliorer la précision mais demandent plus de calcul. t-SNE et UMAP servent à explorer, pas à prouver des groupes réels. Les résultats doivent être confrontés aux contraintes métiers.

**Références :** documentation scikit-learn, UCI Machine Learning Repository, documentation Gymnasium, documentation XGBoost et UMAP. Les schémas sont créés dans ce notebook. **Usage d’IA :** ce notebook a été enrichi avec l’assistance d’un outil d’IA ; les exécutions, interprétations et sources doivent être contrôlées par les auteurs avant la remise.
